#### How to groupby & aggregation using `collect_list() & collect_set()`?

**Array aggregation for denormalization.**

✔️ collect_list() = array with duplicates

✔️ collect_set() = unique array

**Handling Nulls:**
- By default, `collect_list and collect_set` do not include `null values` in the resulting collection.

In [0]:
sales_products = [
 ("Alekya", "Python"),
 ("Alekya", "SQL"),
 ("Alekya", "databricks"),
 ("Alekya", "Python"),
 ("Alekya", None),
 ("Alekya", ""),  
 ("Niroop", "devops"),
 ("Niroop", "devops"), 
 ("Niroop", "VS"),
 ("Niroop", None),
 ("Sandip", "airflow"),
 ("Sandip", "dbt"),
 ("Sandip", ""), 
]

df_collect = spark.createDataFrame(sales_products, ["EmpName", "Technology"])
display(df_collect)

EmpName,Technology
Alekya,Python
Alekya,SQL
Alekya,databricks
Alekya,Python
Alekya,null
Alekya,
Niroop,devops
Niroop,devops
Niroop,VS
Niroop,null


In [0]:
from pyspark.sql.functions import collect_list, collect_set

##### a) Using collect_list()
- `Keeps duplicates`
- `Order is not guaranteed`

In [0]:
# Array aggregations
df_collect_list = df_collect \
    .groupBy("EmpName") \
    .agg(collect_list("Technology").alias("technology_list"))

display(df_collect_list)

EmpName,technology_list
Alekya,"List(Python, SQL, databricks, Python, )"
Niroop,"List(devops, devops, VS)"
Sandip,"List(airflow, dbt, )"


##### b) Using collect_set()
- `Removes duplicates`
- `Order is not guaranteed`

In [0]:
# Array aggregations
df_collect_set = df_collect \
    .groupBy("EmpName") \
    .agg(collect_set("Technology").alias("technology_list"))

display(df_collect_set)

EmpName,technology_list
Alekya,"List(Python, SQL, databricks, )"
Niroop,"List(devops, VS)"
Sandip,"List(airflow, dbt, )"


##### c) Multiple Aggregations

In [0]:
from pyspark.sql.functions import count, countDistinct

df_collect_mlt = df_collect \
    .groupBy("EmpName") \
    .agg(collect_list("Technology").alias("technology_list"),
         count("Technology").alias("tech_cnt"),
         collect_set("Technology").alias("technology_set"),
         countDistinct("Technology").alias("tech_distinct"))

display(df_collect_mlt)

EmpName,technology_list,tech_cnt,technology_set,tech_distinct
Alekya,"List(Python, Python, SQL, databricks, )",5,"List(Python, SQL, databricks, )",4
Niroop,"List(devops, devops, VS)",3,"List(devops, VS)",2
Sandip,"List(airflow, dbt, )",3,"List(airflow, dbt, )",3


##### d) Collect Multiple Columns

In [0]:
from pyspark.sql.functions import struct

data = [("HR", "Nanda", 1000),
        ("HR", "Girish", 2000),
        ("HR", "Darshan", None),
        ("HR", "", 5000),
        ("HR", "Girish", 2000),
        ("IT", "Monica", 4000),        
        ("IT", "Krishna", 3000),
        ("IT", "Suresh", None),
        ("IT", "", 5000),
        ("IT", "Krishna", 3000)
        ]

df_str = spark.createDataFrame(data, ["dept", "name", "salary"])
display(df_str)

dept,name,salary
HR,Nanda,1000
HR,Girish,2000
HR,Darshan,null
HR,,5000
HR,Girish,2000
IT,Monica,4000
IT,Krishna,3000
IT,Suresh,null
IT,,5000
IT,Krishna,3000


In [0]:
df_str \
   .groupBy("dept") \
   .agg(collect_list(struct("name", "salary")).alias("employees")) \
   .display()

dept,employees
HR,"List(List(Nanda, 1000), List(Girish, 2000), List(Darshan, null), List(, 5000), List(Girish, 2000))"
IT,"List(List(Monica, 4000), List(Krishna, 3000), List(Suresh, null), List(, 5000), List(Krishna, 3000))"


In [0]:
df_str \
   .groupBy("dept") \
   .agg(collect_set(struct("name", "salary")).alias("employees")) \
   .display()

dept,employees
HR,"List(List(Nanda, 1000), List(Girish, 2000), List(Darshan, null), List(, 5000))"
IT,"List(List(Monica, 4000), List(Krishna, 3000), List(Suresh, null), List(, 5000))"


##### e) Sorting Collected Data

- `Since order is not guaranteed, use sort_array()`

In [0]:
from pyspark.sql.functions import sort_array

df_str \
  .groupBy("dept") \
  .agg(collect_list("name").alias("collect_list"),
       sort_array(collect_list("name")).alias("sorted_list")) \
  .display()

dept,collect_list,sorted_list
HR,"List(Nanda, Girish, Darshan, , Girish)","List(, Darshan, Girish, Girish, Nanda)"
IT,"List(Monica, Krishna, Suresh, , Krishna)","List(, Krishna, Krishna, Monica, Suresh)"


In [0]:
from pyspark.sql.functions import sort_array

df_str \
  .groupBy("dept") \
  .agg(collect_set("name").alias("collect_list"),
       sort_array(collect_set("name")).alias("sorted_list")) \
  .display()

dept,collect_list,sorted_list
HR,"List(Nanda, Girish, Darshan, )","List(, Darshan, Girish, Nanda)"
IT,"List(Monica, Krishna, Suresh, )","List(, Krishna, Monica, Suresh)"


| Function       | Keeps Duplicates | Removes Duplicates | Order            |
| -------------- | ---------------- | ------------------ | ---------------- |
| collect_list() | ✅ Yes            | ❌ No               | ❌ Not guaranteed |
| collect_set()  | ❌ No             | ✅ Yes              | ❌ Not guaranteed |